# SakiCoder Colab Training (Serious Run)
## Author

Developed by Saqib Sarwar  
Solo developer from Srinagar, J&K, India

This notebook sets up full training workflow on Colab with Drive-backed checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/Saqibsarwar12/sakicoder.git
%cd /content/sakicoder

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Optional: upload a prebuilt training package created locally.
# If no file is uploaded, this cell does nothing.
from google.colab import files
import os, zipfile
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name, 'r') as zf:
            zf.extractall('/content/sakicoder')
        print(f'Extracted {name} into /content/sakicoder')

In [ ]:
# Safe coding dataset preparation pipeline.
!python scripts/collect_coding_sources.py
!python scripts/normalize_code_dataset.py
!python scripts/deduplicate_dataset.py
!python scripts/package_colab_dataset.py

In [ ]:
!python tokenizer/train_tokenizer.py --input_dir data --output tokenizer/tokenizer.json --vocab_size 16000

In [ ]:
!python scripts/build_instruction_dataset.py

In [ ]:
!python scripts/prepare_data.py --input_dir data --tokenizer tokenizer/tokenizer.json --output_dir data/processed --block_size 512 --stride 256 --mode instruct

In [ ]:
!python scripts/train.py --config configs/tiny.json --tokenizer tokenizer/tokenizer.json --data_dir data/processed --out_dir checkpoints/tiny-colab --max_steps 30 --mode instruct

In [ ]:
!python scripts/train.py --config configs/small.json --tokenizer tokenizer/tokenizer.json --data_dir data/processed --out_dir checkpoints/instruct-long --max_steps 2000 --mode instruct

In [ ]:
!mkdir -p /content/drive/MyDrive/sakicoder_checkpoints
!cp -r checkpoints/instruct-long /content/drive/MyDrive/sakicoder_checkpoints/

In [ ]:
# Resume example (rerun later)
!python scripts/train.py --config configs/small.json --tokenizer tokenizer/tokenizer.json --data_dir data/processed --out_dir checkpoints/instruct-long --max_steps 3000 --mode instruct

In [ ]:
!python scripts/generate.py --checkpoint checkpoints/instruct-long/latest.pt --tokenizer tokenizer/tokenizer.json --config configs/small.json --prompt "<|user|>Create a FastAPI endpoint for health checks.<|assistant|>" --max_new_tokens 180 --output outputs/generation_sample.txt

In [ ]:
!zip -r sakicoder_instruct_long.zip checkpoints/instruct-long tokenizer/tokenizer.json outputs/generation_sample.txt